In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load dataset
df = pd.read_csv('diabetic_data.csv')

# Replace '?' string placeholders with proper NaN missing values
df.replace('?', np.nan, inplace=True)
print(f"Initial raw shape: {df.shape}")

Initial raw shape: (101766, 50)


In [3]:
# 1. Exclude death/hospice discharges (discharge_disposition_id 11, 13, 14, 19, 20, 21)
# These patients physically cannot be readmitted, so including them injects label noise.
expired_ids = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(expired_ids)].copy()

# 2. Drop ultra-sparse and non-predictive administrative identifier columns
df.drop(columns=['weight', 'payer_code', 'encounter_id'], inplace=True)

# 3. Re-frame target variable into binary decision task: 
# 1 = Readmitted <30 days (High Risk), 0 = Not readmitted <30 days
df['target'] = (df['readmitted'] == '<30').astype(int)
df.drop(columns=['readmitted'], inplace=True)

print(f"Shape after filtering: {df.shape}")
print("Target balance:\n", df['target'].value_counts(normalize=True))

Shape after filtering: (99343, 47)
Target balance:
 target
0    0.886112
1    0.113888
Name: proportion, dtype: float64


In [4]:
# GroupShuffleSplit ensures all encounters for a specific patient stay together in either Train or Test
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['patient_nbr']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Separate features (X) and target (y), dropping patient_nbr identifier
X_train = train_df.drop(columns=['target', 'patient_nbr'])
y_train = train_df['target']

X_test = test_df.drop(columns=['target', 'patient_nbr'])
y_test = test_df['target']

print(f"Train features shape: {X_train.shape}")
print(f"Test features shape:  {X_test.shape}")

Train features shape: (79541, 45)
Test features shape:  (19802, 45)


In [5]:
def map_icd9(code):
    if pd.isna(code):
        return 'Missing'
    code_str = str(code)
    if code_str.startswith('V') or code_str.startswith('E'):
        return 'Other'
    try:
        val = float(code_str)
        if 390 <= val <= 459 or val == 785:
            return 'Circulatory'
        elif 460 <= val <= 519 or val == 786:
            return 'Respiratory'
        elif 520 <= val <= 579 or val == 787:
            return 'Digestive'
        elif int(val) == 250:
            return 'Diabetes'
        elif 800 <= val <= 999:
            return 'Injury'
        elif 710 <= val <= 739:
            return 'Musculoskeletal'
        elif 580 <= val <= 629 or val == 788:
            return 'Genitourinary'
        elif 140 <= val <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except ValueError:
        return 'Other'

# Apply diagnostic grouping to both splits separately
for col in ['diag_1', 'diag_2', 'diag_3']:
    X_train[col] = X_train[col].apply(map_icd9)
    X_test[col] = X_test[col].apply(map_icd9)

In [6]:
# 1. Cast integer-coded categorical variables to strings
coded_cats = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
for c in coded_cats:
    X_train[c] = X_train[c].astype(str)
    X_test[c] = X_test[c].astype(str)

# 2. Identify numeric and categorical columns
numeric_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

categorical_features = [c for c in X_train.columns if c not in numeric_features]

# 3. Create Scikit-Learn Pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# 4. FIT on training data ONLY, then transform both splits
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Retrieve One-Hot encoded column names
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
encoded_cat_names = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(encoded_cat_names)

# Convert arrays back into DataFrames
X_train_clean = pd.DataFrame(X_train_processed, columns=all_feature_names)
X_test_clean = pd.DataFrame(X_test_processed, columns=all_feature_names)

print("Preprocessing complete!")
print(f"Final Processed X_train shape: {X_train_clean.shape}")
print(f"Final Processed X_test shape:  {X_test_clean.shape}")

Preprocessing complete!
Final Processed X_train shape: (79541, 257)
Final Processed X_test shape:  (19802, 257)


In [6]:
X_train_clean.to_csv('X_train_processed.csv', index=False)
X_test_clean.to_csv('X_test_processed.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Saved clean processed datasets to CSV for Member 3.")

Saved clean processed datasets to CSV for Member 3.


In [7]:
# Feature Construction: Combine previous care utilization into a single total encounter score
X_train['total_visits'] = X_train['number_outpatient'] + X_train['number_emergency'] + X_train['number_inpatient']
X_test['total_visits'] = X_test['number_outpatient'] + X_test['number_emergency'] + X_test['number_inpatient']

# Append 'total_visits' to numeric_features list
if 'total_visits' not in numeric_features:
    numeric_features.append('total_visits')

In [10]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE strictly on transformed training data (Never on test data!)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_clean, y_train)

print("Target balance BEFORE SMOTE:\n", y_train.value_counts())
print("\nTarget balance AFTER SMOTE:\n", y_train_smote.value_counts())

# Export SMOTE-balanced training set alongside standard set
X_train_smote.to_csv('X_train_smote.csv', index=False)
y_train_smote.to_csv('y_train_smote.csv', index=False)
print("SMOTE datasets exported successfully!")

Target balance BEFORE SMOTE:
 target
0    70443
1     9098
Name: count, dtype: int64

Target balance AFTER SMOTE:
 target
0    70443
1    70443
Name: count, dtype: int64
SMOTE datasets exported successfully!


In [9]:
%pip install imbalanced-learn


   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]

Note: you may need to restart the kernel to use updated packages.
